# Transformers Deep Dive — companion notebook

See the full attention matrix, watch the softmax sharpen with input scale, and plot the quadratic cost that drives long-context pricing.

Standard library plus matplotlib. No model, no GPU.

## 1. The mechanism

Four interpretable dimensions so you can check the result by eye: `[animate, place, motion, state]`.

In [1]:
import math

SENTENCE = ["the","animal","did","not","cross","the","street",
            "because","it","was","too","tired"]
BASE = {
    "the":[0.02,0.02,0.02,0.02], "animal":[0.95,0.05,0.20,0.30],
    "did":[0.05,0.02,0.40,0.05], "not":[0.02,0.02,0.10,0.10],
    "cross":[0.10,0.35,0.92,0.05], "street":[0.03,0.94,0.25,0.05],
    "because":[0.02,0.02,0.02,0.15], "it":[0.50,0.45,0.10,0.35],
    "was":[0.05,0.05,0.05,0.30], "too":[0.02,0.02,0.02,0.25],
    "tired":[0.60,0.02,0.05,0.95],
}
W_Q = [[1,0,0,0.9],[0,1,0,0],[0,0,1,0],[0,0,0,1]]
W_K = [[1,0,0,0],[0,1,0,0],[0,0,1,0],[0.9,0,0,1]]
D = 4

def mv(M, v): return [sum(r[i]*v[i] for i in range(len(v))) for r in M]
def softmax(xs):
    m = max(xs); e = [math.exp(x-m) for x in xs]; s = sum(e)
    return [x/s for x in e]

def weights(query, tokens, scale):
    emb = {w: [x*scale for x in v] for w, v in BASE.items()}
    q = mv(W_Q, emb[query])
    sc = [sum(a*b for a, b in zip(q, mv(W_K, emb[t])))/math.sqrt(D) for t in tokens]
    return softmax(sc)

w = weights("it", SENTENCE, 3.0)
for x, t in sorted(zip(w, SENTENCE), reverse=True)[:6]:
    print(f"  {x:.3f}  {t}")

  0.569  animal
  0.235  tired
  0.134  it
  0.022  street
  0.013  cross
  0.005  was


## 2. The full attention matrix

Every token attending to every token — this is the `n x n` object whose size drives the quadratic cost.

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

M = [weights(t, SENTENCE, 3.0) for t in SENTENCE]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(M, cmap="viridis")
ax.set_xticks(range(len(SENTENCE))); ax.set_xticklabels(SENTENCE, rotation=90)
ax.set_yticks(range(len(SENTENCE))); ax.set_yticklabels(SENTENCE)
ax.set_xlabel("attended TO"); ax.set_ylabel("attending FROM")
ax.set_title("Attention weights, one head")
fig.colorbar(im, ax=ax, shrink=.8)
plt.tight_layout(); plt.show()

row = M[SENTENCE.index("it")]
print("row for 'it':")
for t, x in sorted(zip(SENTENCE, row), key=lambda kv: -kv[1])[:4]:
    print(f"  {x:.3f}  {t}")

row for 'it':
  0.569  animal
  0.235  tired
  0.134  it
  0.022  street


## 3. Input scale decides whether attention is sharp or mush

The article notes this in passing; it is worth seeing. Small vectors give a nearly uniform softmax and the mechanism looks like it does nothing.

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

scales = [s/10 for s in range(5, 61, 1)]
ratio, peak = [], []
for s in scales:
    w = weights("it", SENTENCE, s)
    d = dict(zip(SENTENCE, w))
    ratio.append(d["animal"]/d["street"])
    peak.append(max(w))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(scales, ratio, label="animal / street weight ratio")
ax.axhline(1.0, ls="--", lw=.8, color="grey")
ax.set_xlabel("embedding scale"); ax.set_ylabel("ratio")
ax.set_yscale("log"); ax.grid(alpha=.3)
ax2 = ax.twinx(); ax2.plot(scales, peak, color="tab:orange")
ax2.set_ylabel("largest single weight", color="tab:orange")
ax.set_title("Small vectors -> uniform softmax -> no visible attention")
plt.tight_layout(); plt.show()

for s in (0.5, 1.0, 3.0, 6.0):
    d = dict(zip(SENTENCE, weights("it", SENTENCE, s)))
    print(f"  scale {s:>4}: animal {d['animal']:.3f}  street {d['street']:.3f}  "
          f"ratio {d['animal']/d['street']:>6.1f}x")

  scale  0.5: animal 0.092  street 0.084  ratio    1.1x
  scale  1.0: animal 0.123  street 0.086  ratio    1.4x
  scale  3.0: animal 0.569  street 0.022  ratio   25.7x
  scale  6.0: animal 0.969  street 0.000  ratio 437486.0x


## 4. The quadratic term, in money

Plug in your own per-token price and see where long context stops being free.

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ns = [2**i for i in range(6, 18)]
pairs = [n*n for n in ns]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns, pairs, marker="o", ms=3, label="attention pair comparisons  O(n^2)")
ax.plot(ns, ns, marker="s", ms=3, label="feed-forward work  O(n)")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("sequence length (tokens)"); ax.set_ylabel("relative work")
ax.legend(); ax.grid(alpha=.3)
ax.set_title("Double the context, quadruple the attention")
plt.tight_layout(); plt.show()

print(f"  {'tokens':>8} {'pairs':>15} {'vs 1k baseline':>16}")
base = 1000*1000
for n in (1_000, 4_000, 16_000, 128_000, 1_000_000):
    print(f"  {n:>8,} {n*n:>15,} {n*n/base:>15,.0f}x")
print("\nchunking n into k pieces: k * (n/k)^2 = n^2/k  -> a factor-of-k saving")

    tokens           pairs   vs 1k baseline
     1,000       1,000,000               1x
     4,000      16,000,000              16x
    16,000     256,000,000             256x
   128,000  16,384,000,000          16,384x
  1,000,000 1,000,000,000,000       1,000,000x

chunking n into k pieces: k * (n/k)^2 = n^2/k  -> a factor-of-k saving


## 5. Your turn

1. Change the vector for `it` toward `place` and watch the matrix row move.
2. Add a second head with different `W_Q`/`W_K` and see whether it tracks a different relationship.
3. Put your real per-token price into §4 and find the context length where attention cost overtakes everything else in your budget.